# Challenge 4: Fix and compare to baseline

Evals earn their keep when they drive changes. The loop:

1. Run evals on the current agent. (Challenge 3 was this.)
2. Read the failures and form a hypothesis.
3. Change **one thing**.
4. Rerun the **same** evals on the **same** dataset version and compare against the baseline.

**One cell was blank in the workshop. Here it is filled in, so every cell runs.**

In [ ]:
import os
import sys

for path in ("..", "../lab"):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from langsmith import Client, uuid7

import lab_eval
from eval_lab.agent import SYSTEM_PROMPT_V1, TOOLS, build_agent, extract_sources, extract_tool_calls

client = Client()
DATASET = lab_eval.dataset_name()
print("project:", os.environ["LANGSMITH_PROJECT"])
print("dataset:", DATASET)

from eval_lab.evaluators import groundedness, keyword_match, make_run_agent, tool_selection

The evaluators and run function are the ones you wrote in challenges 2 and 3, imported so the baseline and the fix are scored by identical code.

## Error analysis

Scores say how often the agent fails, not why. Look at every failing row of your `v1-baseline` experiment, name the cause, and count the causes. The biggest count is what to fix first.

In [ ]:
lab_eval.failure_modes("v1-baseline")

Read the counts. If you see a different pattern from the one described below, trust your table: model behavior varies from run to run.

## The hypothesis

If `ungrounded answer` and `skipped search_confluence` lead: the prompt told the agent to answer from its quick reference without searching. The fix is a prompt change: search first, answer only from the results, cite the document id, and say so when nothing is found.

The next cell holds this code:

```python
SYSTEM_PROMPT_V2 = (
    "You are an internal assistant for employees. Follow this procedure exactly.\n"
    "1. For any policy or process question, call search_confluence first. "
    "Answer only from the documents it returns, and cite the document id, for example [KB-101].\n"
    "2. If the search returns no matching documents, say you could not find it in the "
    "knowledge base. Do not guess or answer from general knowledge.\n"
    "3. For questions about a ticket, call lookup_jira.\n"
    "4. Call update_jira when the user asks you to add a comment to a ticket or update it, "
    "and only after lookup_jira."
)
```

In [ ]:
# Solution. In the workshop this cell was blank and you wrote it.
SYSTEM_PROMPT_V2 = (
    "You are an internal assistant for employees. Follow this procedure exactly.\n"
    "1. For any policy or process question, call search_confluence first. "
    "Answer only from the documents it returns, and cite the document id, for example [KB-101].\n"
    "2. If the search returns no matching documents, say you could not find it in the "
    "knowledge base. Do not guess or answer from general knowledge.\n"
    "3. For questions about a ticket, call lookup_jira.\n"
    "4. Call update_jira when the user asks you to add a comment to a ticket or update it, "
    "and only after lookup_jira."
)

## Rerun the same evals

Only the system prompt changes. Same dataset version, same evaluators, new experiment prefix `v2-fixed`.

In [ ]:
agent_v2 = build_agent(SYSTEM_PROMPT_V2)

results = client.evaluate(
    make_run_agent(agent_v2),
    data=lab_eval.pinned_examples(client),
    evaluators=[groundedness, tool_selection, keyword_match],
    experiment_prefix="v2-fixed",
    max_concurrency=4,
)
print("experiment:", results.experiment_name)

## Compare against the baseline

One row per experiment on this dataset. Read down the `groundedness` and `tool_selection` columns: `v1-baseline` versus `v2-fixed`.

In [ ]:
lab_eval.compare(prefixes=("first-eval", "v1-baseline", "v2-fixed"),
                 keys=["keyword_match", "groundedness", "tool_selection"])

Run error analysis again, on the fixed experiment, to see what is left.

In [ ]:
lab_eval.failure_modes("v2-fixed")

**Result:** `v2-fixed` scores higher than `v1-baseline` on `groundedness` and `tool_selection`. For a side-by-side view, open your dataset in LangSmith, select both experiments, and click **Compare**: each row shows which answers got better.

Two cautions. LLM output varies, so rerun if one row flips: one row of 8 moves a score by 0.125. And a higher average on 8 examples is a signal, not proof: a real dataset grows every time production finds a new failure.